# Sensitivity to downsampling
Keep feature costs, four cutoffs and k fixed within each case. Only the requested sampling ratio changes. This tutorial reruns all six ratios without a hard upper cap; it is distinct from the paper's 17 archived runs plus one supplementary Deep-STARmap run. Use the separate figure notebook for the exact archived series.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', './prepared_inputs')).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
# Set DATA or RAFTUP_TUTORIAL_DATA to your prepared input directory.
# pair.json contains file paths, exact cost-matrix cell order and spatial calibration.
a, b, cost, cfg = load_pair(DATA / 'pair.json')
print(a.shape, b.shape, cost.shape)


In [ ]:
import pandas as pd
RATIOS = [.02, .04, .05, .067, .08, .10]
# Select the documented protocol matching your input pair.
PROTOCOLS = {
 'dlpfc_0708': dict(feature_cutoff_downsample=.10, spatial_cutoff_downsample_um=400,
                    feature_cutoff_recovery=.20, spatial_cutoff_recovery_um=200, k=20),
 'merfish_0409': dict(feature_cutoff_downsample=.20, spatial_cutoff_downsample_um=100,
                    feature_cutoff_recovery=.25, spatial_cutoff_recovery_um=60, k=10),
 'deep_starmap_13_14': dict(feature_cutoff_downsample=.20, spatial_cutoff_downsample_um=100,
                    feature_cutoff_recovery=.25, spatial_cutoff_recovery_um=60, k=6),
}
parameters = PROTOCOLS[cfg['name']]
rows = []
for ratio in RATIOS:
    P, decision = run_manual_alignment(a, b, cost,
        technology=cfg['data_technology'], pixels_per_100um=cfg['pixels_per_100um'],
        downsampling_ratio=ratio, random_state=42,
        min_downsampled_points=2, max_downsampled_points=None, **parameters)
    save_alignment(P, decision, OUT / 'downsampling' / cfg['name'] / f'ratio_{ratio:.4f}')
    rows.append(dict(ratio=ratio, **coverage(P)))
display(pd.DataFrame(rows))


Runtime definitions and historical LAA/GPR results are retained in `protocol_sources/` and `figures/04_downsampling.ipynb`. This is a single-seed sensitivity experiment, not a random-seed variance estimate.
